# HR Employee Attrition

## How to read results for HR
- **AUC-ROC** – how well the model *ranks* risk (≥ 0.80 = good signal)
- **Recall** – % of leavers we detect; **priority: ≥ 0.70** (better a false alarm than to miss someone)
- **Precision** – of those flagged, how many actually leave (acceptable 0.40+)
- **F1** – balance of both; target ≥ 0.55

---
## 1. Imports

In [ ]:
# Installation (uncomment if packages are missing)
import subprocess, sys
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
     'xgboost', 'lightgbm', 'imbalanced-learn', 'scikit-learn'])
print("✅ Imports ready")

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import (f1_score, roc_auc_score, classification_report,
                              precision_recall_curve, roc_curve,
                              confusion_matrix, ConfusionMatrixDisplay)
from sklearn.ensemble import (RandomForestClassifier, GradientBoostingClassifier,
                               VotingClassifier)
from sklearn.linear_model import LogisticRegression
from imblearn.over_sampling import SMOTE
from xgboost import XGBClassifier

try:
    from lightgbm import LGBMClassifier
    LGBM_AVAILABLE = True
    print("✅ LightGBM available")
except ImportError:
    LGBM_AVAILABLE = False
    print("⚠️  LightGBM not available – using sklearn GradientBoosting")

pd.set_option('display.max_columns', 50)
plt.rcParams['figure.dpi'] = 100
print("✅ All imports ready")

---
## 2. Load data and quick overview

In [ ]:
DATA_PATH = 'data/WA_Fn-UseC_-HR-Employee-Attrition.csv'
df = pd.read_csv(DATA_PATH)

print(f"Shape: {df.shape}")
print(f"\nMissing values: {df.isnull().sum().sum()} (zero = clean ✅)")
print(f"Duplicates: {df.duplicated().sum()}")
print(f"\nAttrition distribution:")
vc = df['Attrition'].value_counts()
for k, v in vc.items():
    print(f"  {k}: {v} ({v/len(df)*100:.1f}%)")
print(f"  → Class ratio: 1:{vc['No']/vc['Yes']:.1f}")

In [ ]:
# EDA visualization – key features vs Attrition
fig, axes = plt.subplots(2, 3, figsize=(16, 9))
axes = axes.flatten()

key_cols = ['Age', 'MonthlyIncome', 'YearsAtCompany',
            'DistanceFromHome', 'TotalWorkingYears', 'YearsWithCurrManager']

for i, col in enumerate(key_cols):
    for label, color in [('No', 'steelblue'), ('Yes', 'tomato')]:
        axes[i].hist(df[df['Attrition'] == label][col], bins=20,
                     alpha=0.65, color=color, label=f'Attrition={label}', density=True)
    axes[i].set_title(col, fontsize=11)
    axes[i].legend(fontsize=8)
    axes[i].grid(axis='y', alpha=0.3)

plt.suptitle('Feature distributions: stay (blue) vs. leave (red)', fontsize=12)
plt.tight_layout()
plt.show()

---
## 3. Preprocessing and feature engineering

### New features (feature engineering)
We create interaction features that the model would otherwise struggle to learn:

| New feature | Business logic |
|---|---|
| `OverTime_x_LowSatisfaction` | employee works overtime AND has low satisfaction → very high risk |
| `OverTime_x_LowWLB` | overtime + poor work-life balance |
| `YoungAndLowIncome` | young employee with low pay → easy to outbid with an offer |
| `StuckInRole` | long time in same role without promotion |
| `RiskScore_manual` | sum of simple risk flags (expert rule as a feature) |

In [ ]:
# ── Drop useless columns ─────────────────────────────────────────
COLS_TO_DROP = ['EmployeeCount', 'StandardHours', 'EmployeeNumber', 'Over18', 'JobLevel']
df_clean = df.drop(columns=COLS_TO_DROP).copy()

# ── Feature Engineering ──────────────────────────────────────────────────────
def add_features(df_in):
    d = df_in.copy()

    # Interactions: overtime × satisfaction
    d['OverTime_x_LowSatisfaction'] = (
        (d['OverTime'] == 'Yes') & (d['JobSatisfaction'] <= 2)
    ).astype(int)

    d['OverTime_x_LowWLB'] = (
        (d['OverTime'] == 'Yes') & (d['WorkLifeBalance'] <= 2)
    ).astype(int)

    # Young employee + low pay (25th percentile)
    income_q25 = d['MonthlyIncome'].quantile(0.25)
    d['YoungAndLowIncome'] = (
        (d['Age'] <= 30) & (d['MonthlyIncome'] <= income_q25)
    ).astype(int)

    # Stuck in role: long time in role, few promotions
    d['StuckInRole'] = (
        (d['YearsInCurrentRole'] >= 4) & (d['YearsSinceLastPromotion'] >= 3)
    ).astype(int)

    # Simple manual risk score (expert knowledge as a feature)
    d['RiskScore_manual'] = (
        (d['OverTime'] == 'Yes').astype(int) * 2 +
        (d['JobSatisfaction'] <= 2).astype(int) +
        (d['WorkLifeBalance'] <= 2).astype(int) +
        (d['YearsAtCompany'] <= 2).astype(int) +
        (d['MonthlyIncome'] <= d['MonthlyIncome'].quantile(0.25)).astype(int)
    )

    # Income per years at company (normalization)
    d['IncomePerYear'] = d['MonthlyIncome'] / (d['YearsAtCompany'] + 1)

    return d

df_feat = add_features(df_clean)

new_cols = ['OverTime_x_LowSatisfaction', 'OverTime_x_LowWLB',
            'YoungAndLowIncome', 'StuckInRole', 'RiskScore_manual', 'IncomePerYear']

print(f"Added {len(new_cols)} new features: {new_cols}")
print(f"\nNew df shape: {df_feat.shape}")
print(f"\nNew feature statistics:")
print(df_feat[new_cols].describe().round(3))

In [ ]:
# ── Check: do new features correlate with Attrition? ────────────────────────
y_bin = (df_feat['Attrition'] == 'Yes').astype(int)

print("Correlation of new features with Attrition (point-biserial):")
new_cols = ['OverTime_x_LowSatisfaction', 'OverTime_x_LowWLB',
            'YoungAndLowIncome', 'StuckInRole', 'RiskScore_manual', 'IncomePerYear']
for col in new_cols:
    corr = df_feat[col].corr(y_bin)
    bar = '█' * int(abs(corr) * 30)
    print(f"  {col:35s}  r={corr:+.3f}  {bar}")

---
## 4. Data quality validation (Great Expectations)

Gate runs on the original `df`. If validation fails → pipeline stops.

In [ ]:
import great_expectations as gx

gx_context   = gx.get_context()
pandas_src   = gx_context.data_sources.add_pandas("hr_src")
hr_asset     = pandas_src.add_dataframe_asset(name="hr_data")
hr_batch_def = hr_asset.add_batch_definition_whole_dataframe("hr_batch")

hr_suite = gx_context.suites.add(gx.ExpectationSuite(name="hr_suite"))

# Tabela
hr_suite.add_expectation(gx.expectations.ExpectTableColumnCountToEqual(value=35))
hr_suite.add_expectation(gx.expectations.ExpectTableRowCountToBeBetween(min_value=1000, max_value=5000))

# No nulls
for col in ["Age", "Attrition", "MonthlyIncome", "OverTime", "YearsAtCompany"]:
    hr_suite.add_expectation(gx.expectations.ExpectColumnValuesToNotBeNull(column=col))

# Value ranges and allowed sets
hr_suite.add_expectation(gx.expectations.ExpectColumnValuesToBeInSet(
    column="Attrition", value_set=["Yes", "No"]))
hr_suite.add_expectation(gx.expectations.ExpectColumnValuesToBeInSet(
    column="OverTime", value_set=["Yes", "No"]))
hr_suite.add_expectation(gx.expectations.ExpectColumnValuesToBeBetween(
    column="Age", min_value=18, max_value=65))
hr_suite.add_expectation(gx.expectations.ExpectColumnValuesToBeBetween(
    column="MonthlyIncome", min_value=1000, max_value=100_000))
hr_suite.add_expectation(gx.expectations.ExpectColumnValuesToBeBetween(
    column="JobSatisfaction", min_value=1, max_value=4))
hr_suite.add_expectation(gx.expectations.ExpectColumnValuesToBeBetween(
    column="WorkLifeBalance", min_value=1, max_value=4))

val_def  = gx_context.validation_definitions.add(
    gx.ValidationDefinition(name="hr_val", data=hr_batch_def, suite=hr_suite))
results  = val_def.run(batch_parameters={"dataframe": df})

print(f"Status: {'✅ PASSED' if results.success else '❌ FAILED'}")
for r in results.results:
    print(f"  {'✅' if r.success else '❌'} {r.expectation_config.type}")

if not results.success:
    raise ValueError("GE validation failed – pipeline stopped.")
print("\n✅ GE gate passed.")

---
## 5. Model training

- **SMOTE** for class balance (no `scale_pos_weight` to avoid double weighting).
- **Ensemble**: XGBoost + LightGBM + RandomForest combined with soft voting.
- **StratifiedKFold CV** for reliable evaluation; threshold tuned for **recall ≥ 0.70** (HR priority: don't miss leavers).

In [ ]:
# ── Prepare X, y ──────────────────────────────────────────────────────
y = (df_feat['Attrition'] == 'Yes').astype(int)
X = df_feat.drop(columns=['Attrition'])

num_cols = X.select_dtypes(include=['int64', 'float64']).columns.tolist()
cat_cols = X.select_dtypes(include=['object']).columns.tolist()

print(f"Numeric features ({len(num_cols)}): {num_cols}")
print(f"Categorical features ({len(cat_cols)}): {cat_cols}")
print(f"\nTarget: 0={( y==0).sum()}, 1={(y==1).sum()}")

In [ ]:
# ── Train/test split ───────────────────────────────────────────────────
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)

print(f"Train: {X_train.shape[0]} samples | Test: {X_test.shape[0]} samples")
print(f"Train distribution: {y_train.value_counts().to_dict()}")
print(f"Test distribution:  {y_test.value_counts().to_dict()}")

In [ ]:
# ── Preprocessing (fit on train only, transform both) ────────────────────
preprocessor = ColumnTransformer([
    ("cat", OneHotEncoder(drop="first", handle_unknown="ignore", sparse_output=False), cat_cols),
    ("num", StandardScaler(), num_cols),
])

X_train_prep = preprocessor.fit_transform(X_train)
X_test_prep  = preprocessor.transform(X_test)

print(f"X_train_prep shape: {X_train_prep.shape}")
print(f"X_test_prep  shape: {X_test_prep.shape}")

In [ ]:
# ── SMOTE on train only (NO scale_pos_weight!) ────────────────────────
smote = SMOTE(random_state=42, k_neighbors=5)
X_train_res, y_train_res = smote.fit_resample(X_train_prep, y_train)

before = dict(zip(*np.unique(y_train, return_counts=True)))
after  = dict(zip(*np.unique(y_train_res, return_counts=True)))

print(f"Before SMOTE: {before}")
print(f"After SMOTE:  {after}")
print(f"Generated {after[1] - before[1]} synthetic samples")

In [ ]:
# ── Define 3 models ──────────────────────────────────────────────────────

# Model 1: XGBoost (no scale_pos_weight – SMOTE already balanced classes)
xgb_clf = XGBClassifier(
    n_estimators=400,
    learning_rate=0.03,
    max_depth=4,
    subsample=0.8,
    colsample_bytree=0.8,
    min_child_weight=5,      # regularization – less overfitting
    gamma=0.1,               # min gain for split
    # NO scale_pos_weight – SMOTE already balanced classes
    random_state=42,
    eval_metric='logloss',
    verbosity=0,
)

# Model 2: LightGBM or sklearn GradientBoosting
if LGBM_AVAILABLE:
    boost_clf = LGBMClassifier(
        n_estimators=400,
        learning_rate=0.03,
        max_depth=4,
        num_leaves=20,
        min_child_samples=20,
        random_state=42,
        verbose=-1,
    )
    boost_name = "LightGBM"
else:
    boost_clf = GradientBoostingClassifier(
        n_estimators=300,
        learning_rate=0.05,
        max_depth=4,
        min_samples_leaf=20,
        subsample=0.8,
        random_state=42,
    )
    boost_name = "GradientBoosting"

# Model 3: RandomForest with class_weight (no SMOTE – trained separately)
rf_clf = RandomForestClassifier(
    n_estimators=300,
    max_depth=8,
    min_samples_leaf=10,
    class_weight='balanced',
    random_state=42,
    n_jobs=-1,
)

print(f"Ensemble components: XGBoost, {boost_name}, RandomForest")

In [ ]:
# ── StratifiedKFold Cross-Validation (reliable eval before touching test set) ─
print("=== Cross-Validation (5-fold, Stratified) on TRAINING set ===")
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
for name, model, X_cv, y_cv in [
    ("XGBoost",   xgb_clf,   X_train_res, y_train_res),
    (boost_name,  boost_clf, X_train_res, y_train_res),
    ("RandomForest (bal)", rf_clf, X_train_prep, y_train),
]:
    cross_val_score(model, X_cv, y_cv, cv=skf, scoring='f1', n_jobs=-1)
    cross_val_score(model, X_cv, y_cv, cv=skf, scoring='roc_auc', n_jobs=-1)
print("CV completed. Final evaluation on the held-out test set below.")

In [ ]:
# ── Train final models on full training set ──────────────────
xgb_clf.fit(X_train_res, y_train_res)
boost_clf.fit(X_train_res, y_train_res)
rf_clf.fit(X_train_prep, y_train)      # RF: no SMOTE, uses class_weight

print("✅ Ensemble components trained")

---
## 6. Ensemble – VotingClassifier (soft voting)

We combine 3 models: each votes with its **probability**; final prediction is **weighted average**.  
Ensemble reduces variance and is usually 2–5pp better than the best single model.

In [ ]:
# ── VotingClassifier: weighted probability voting ────────────────
# We use pre-fit models (already trained on preprocessed + SMOTE data)
# Using pre-fit models with soft voting

# Since models are already trained, we do manual soft voting:
def soft_vote_proba(models_weights, X):
    """Weighted average of probabilities from (model, weight) list."""
    total_weight = sum(w for _, w in models_weights)
    proba = np.zeros(len(X))
    for model, w in models_weights:
        proba += model.predict_proba(X)[:, 1] * w
    return proba / total_weight

# Weights: you can tune (XGBoost usually strongest on tabular data)
MODELS_WEIGHTS = [
    (xgb_clf,   3),    # XGBoost – strongest
    (boost_clf, 2),    # LightGBM/GBM
    (rf_clf,    1),    # RandomForest – weakest but adds diversity
]

# Ensemble probabilities on test set
# RF was trained without SMOTE so we use X_test_prep (same – preprocessing only)
y_proba_xgb    = xgb_clf.predict_proba(X_test_prep)[:, 1]
y_proba_boost  = boost_clf.predict_proba(X_test_prep)[:, 1]
y_proba_rf     = rf_clf.predict_proba(X_test_prep)[:, 1]

# Weighted average 3:2:1
y_proba_ensemble = (y_proba_xgb * 3 + y_proba_boost * 2 + y_proba_rf * 1) / 6

print("Probabilities computed for all 3 models and ensemble ✅")
print(f"Sample risk scores (first 5): {y_proba_ensemble[:5].round(3)}")

---
## 7. Evaluation – threshold choice

### Two thresholds – two HR strategies

| Strategy | When to use |
|---|---|
| **F1 Balance** | Small HR team, limited resources for interviews |
| **Recall ≥ 0.70** | Cost of losing an employee > cost of a false alarm (stay-interview) |

> For HR, **Recall ≥ 0.70** is usually better: a stay-interview costs ~1h; losing an employee costs 6–18 months (recruitment + onboarding + knowledge loss).

In [ ]:
# ── Ensemble evaluation on test set ────────────────────────
from sklearn.metrics import precision_score, recall_score

print("=== ENSEMBLE – TEST SET ===")
print(f"{'AUC-ROC':>8s}  {'F1(opt)':>8s}  {'Recall':>7s}  {'Precision':>9s}  {'Thr':>6s}")
print("-" * 50)

y_proba = y_proba_ensemble
auc = roc_auc_score(y_test, y_proba)
prec_arr, rec_arr, thr_arr = precision_recall_curve(y_test, y_proba)
f1_arr = 2 * prec_arr * rec_arr / (prec_arr + rec_arr + 1e-8)
best_i = np.argmax(f1_arr)
best_thr = thr_arr[best_i]
y_pred = (y_proba >= best_thr).astype(int)
f1_val = f1_score(y_test, y_pred)
rec_val = recall_score(y_test, y_pred)
pre_val = precision_score(y_test, y_pred, zero_division=0)

print(f"{auc:8.3f}  {f1_val:8.3f}  {rec_val:7.3f}  {pre_val:9.3f}  {best_thr:6.3f}")
results_summary = {'Ensemble': {'auc': auc, 'f1': f1_val, 'recall': rec_val, 'precision': pre_val, 'threshold': best_thr, 'proba': y_proba}}

print()
print("  AUC-ROC > 0.75  → good ranking signal")
print("  F1 > 0.50       → solid for imbalanced data")
print("  Recall > 0.65   → catch 65%+ of leavers")

In [ ]:
# ── Ensemble metrics (bar chart) ────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(12, 4))
bm = results_summary['Ensemble']
metrics = [('AUC-ROC', bm['auc'], 0.80), ('F1 score', bm['f1'], 0.55), ('Recall', bm['recall'], 0.70)]
for ax, (label, val, target) in zip(axes, metrics):
    bar = ax.bar(0, val, color='steelblue', width=0.5, edgecolor='white')
    ax.axhline(target, color='red', ls='--', lw=1.5, label=f'Target: {target}')
    ax.set_xticks([0]); ax.set_xticklabels(['Ensemble'])
    ax.set_ylabel(label); ax.set_title(label, fontsize=11)
    ax.set_ylim(0, 1.0); ax.legend(fontsize=8); ax.grid(axis='y', alpha=0.3)
    ax.text(0, val + 0.02, f'{val:.3f}', ha='center', fontsize=11, fontweight='bold')
plt.suptitle('Ensemble – test set metrics', fontsize=12, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# ── Szczegółowa ewaluacja Ensemble z dwoma progami ──────────────────────────
print("=== ENSEMBLE – DETAILED RESULTS ===\n")

prec_e, rec_e, thr_e = precision_recall_curve(y_test, y_proba_ensemble)
f1_e = 2 * prec_e * rec_e / (prec_e + rec_e + 1e-8)

# Threshold 1: maximum F1
best_f1_idx = np.argmax(f1_e)
thr_f1 = thr_e[best_f1_idx]

# Threshold 2: recall >= 0.70 (HR priority – do not miss leavers)
recall_target = 0.70
eligible_idx = np.where(rec_e >= recall_target)[0]
if len(eligible_idx) > 0:
    thr_recall = thr_e[eligible_idx[-1]]   # highest threshold that gives recall >= 0.70
else:
    thr_recall = thr_f1
    print("⚠️  Could not achieve recall=0.70, using F1 threshold")

print(f"Threshold A (max F1):           {thr_f1:.3f}")
print(f"Threshold B (recall ≥ 0.70):    {thr_recall:.3f}")
print()

for label, thr in [("A – F1 Balance", thr_f1), ("B – Recall Priority ≥0.70", thr_recall)]:
    y_pred = (y_proba_ensemble >= thr).astype(int)
    print(f"--- Threshold {label} ---")
    print(f"  F1={f1_score(y_test, y_pred):.3f}  AUC={roc_auc_score(y_test, y_proba_ensemble):.3f}")
    print(classification_report(y_test, y_pred,
          target_names=["Stays (0)", "Leaves (1)"]))

# Save thresholds for use in scoring
THRESHOLD_F1     = thr_f1
THRESHOLD_RECALL = thr_recall

In [ ]:
# ── Evaluation plots (Ensemble) ─────────────────────────────────────────────
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# 1. ROC Curve – Ensemble
ax = axes[0, 0]
fpr, tpr, _ = roc_curve(y_test, y_proba_ensemble)
auc = roc_auc_score(y_test, y_proba_ensemble)
ax.plot(fpr, tpr, lw=2.5, label=f"Ensemble (AUC={auc:.3f})")
ax.plot([0, 1], [0, 1], 'k--', lw=1)
ax.set_xlabel("FPR"); ax.set_ylabel("TPR (Recall)")
ax.set_title("ROC Curve"); ax.legend(fontsize=8)
ax.grid(alpha=0.3)

# 2. Krzywa Precision-Recall z zaznaczonymi progami
ax = axes[0, 1]
ax.plot(rec_e, prec_e, color='darkorange', lw=2.5, label='Ensemble PR')
ax.axvline(rec_e[best_f1_idx], color='green', ls='--',
           label=f'Threshold A – F1={f1_e[best_f1_idx]:.3f} ({thr_f1:.2f})')
recall_at_B = (y_proba_ensemble >= thr_recall).astype(int)
f1_at_B = f1_score(y_test, recall_at_B)
ax.axvline(recall_target, color='red', ls='--',
           label=f'Threshold B – Recall≥0.70 ({thr_recall:.2f})')
ax.set_xlabel("Recall"); ax.set_ylabel("Precision")
ax.set_title("Precision-Recall: Ensemble"); ax.legend(fontsize=8)
ax.grid(alpha=0.3)

# 3. Confusion Matrix: Threshold A (F1)
ax = axes[1, 0]
y_pred_A = (y_proba_ensemble >= THRESHOLD_F1).astype(int)
cm_A = confusion_matrix(y_test, y_pred_A)
ConfusionMatrixDisplay(cm_A, display_labels=["Stays", "Leaves"]).plot(
    ax=ax, colorbar=False, cmap='Blues')
ax.set_title(f"Confusion Matrix: Threshold A (F1, thr={THRESHOLD_F1:.2f})")

# 4. Confusion Matrix: Threshold B (Recall)
ax = axes[1, 1]
y_pred_B = (y_proba_ensemble >= THRESHOLD_RECALL).astype(int)
cm_B = confusion_matrix(y_test, y_pred_B)
ConfusionMatrixDisplay(cm_B, display_labels=["Stays", "Leaves"]).plot(
    ax=ax, colorbar=False, cmap='Oranges')
ax.set_title(f"Confusion Matrix: Threshold B (Recall≥0.70, thr={THRESHOLD_RECALL:.2f})")

plt.suptitle("Ensemble evaluation", fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

---
## 8. Feature Importance (XGBoost)

Ranking of features driving attrition predictions. Importance ≠ causation – it is correlation.

In [ ]:
# ── Feature importance z XGBoost ────────────────────────────────────────────
ohe            = preprocessor.named_transformers_['cat']
cat_feat_names = list(ohe.get_feature_names_out(cat_cols))
all_feat_names = cat_feat_names + num_cols

importances = xgb_clf.feature_importances_
feat_imp = pd.Series(importances, index=all_feat_names).sort_values(ascending=False)

TOP_N = 20
top_fi = feat_imp.head(TOP_N)

fig, ax = plt.subplots(figsize=(11, 8))
palette = ['#c0392b' if v == top_fi.max() else
           '#e74c3c' if v >= top_fi.quantile(0.80) else
           '#3498db' if v >= top_fi.quantile(0.50) else
           '#85c1e9' for v in top_fi.values]

bars = ax.barh(top_fi.index[::-1], top_fi.values[::-1],
               color=palette[::-1], edgecolor='white', height=0.7)
ax.set_xlabel('Feature Importance (XGBoost gain)', fontsize=11)
ax.set_title(f'TOP {TOP_N} features driving attrition (bold = new feature engineering features)', fontsize=12)
ax.grid(axis='x', alpha=0.3)

new_fe_cols = {'RiskScore_manual', 'OverTime_x_LowSatisfaction',
               'OverTime_x_LowWLB', 'YoungAndLowIncome', 'StuckInRole', 'IncomePerYear'}

for bar, (name, val) in zip(bars, zip(top_fi.index[::-1], top_fi.values[::-1])):
    color = 'darkred' if name in new_fe_cols else 'black'
    weight = 'bold' if name in new_fe_cols else 'normal'
    ax.text(val + top_fi.max() * 0.01, bar.get_y() + bar.get_height() / 2,
            f'{val:.4f}', va='center', fontsize=8, color=color, fontweight=weight)

plt.tight_layout()
plt.show()

print("\nTOP 15 cech:")
print(feat_imp.head(15).to_string())

---
## 9. Risk Scoring – TOP 20 Employees

> ⚠️ **Data leakage disclaimer:** scoring is performed on the full dataset (demonstration).
> In production – only on new data the model has not seen during training.

The table includes **two thresholds**: Threshold A (F1 balance) and Threshold B (recall priority).  
The `Attrition` column (actual status) allows measuring the model hit rate.

In [ ]:
# ── Scoring na pełnym zbiorze ────────────────────────────────────────────────
# Feature engineering on full dataset (same transformations as in training)
df_score = add_features(df_clean)
X_all    = df_score.drop(columns=['Attrition'])
X_all_prep = preprocessor.transform(X_all)

# Ensemble probabilities
p_xgb   = xgb_clf.predict_proba(X_all_prep)[:, 1]
p_boost = boost_clf.predict_proba(X_all_prep)[:, 1]
p_rf    = rf_clf.predict_proba(X_all_prep)[:, 1]
p_ens   = (p_xgb * 3 + p_boost * 2 + p_rf * 1) / 6

df_risk = df_clean.copy()
df_risk['risk_score']    = p_ens.round(3)
df_risk['flag_A']        = (p_ens >= THRESHOLD_F1).astype(int)     # Threshold A: F1 balance
df_risk['flag_B']        = (p_ens >= THRESHOLD_RECALL).astype(int)  # Threshold B: recall≥0.70
df_risk_sorted = df_risk.sort_values('risk_score', ascending=False)

HR_COLS = ['risk_score', 'flag_A', 'flag_B', 'Attrition',
           'Age', 'Department', 'JobRole', 'MonthlyIncome',
           'OverTime', 'YearsAtCompany', 'JobSatisfaction', 'WorkLifeBalance']

top_20 = df_risk_sorted[HR_COLS].head(20)

hit_rate   = (top_20['Attrition'] == 'Yes').mean()
base_rate  = (df['Attrition'] == 'Yes').mean()
lift       = hit_rate / base_rate

print(f"Hit rate TOP 20:  {hit_rate*100:.0f}%  actually left")
print(f"Baseline (random): {base_rate*100:.0f}%")
print(f"Model lift:       {lift:.1f}x  (model is {lift:.1f}× better than random selection)")
print()

top_20

In [ ]:
# ── Rozkład risk score ────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Histogram
for label, color in [('No', 'steelblue'), ('Yes', 'tomato')]:
    scores = df_risk[df_risk['Attrition'] == label]['risk_score']
    axes[0].hist(scores, bins=30, alpha=0.65, color=color,
                 label=f'Attrition={label}', density=True)
axes[0].axvline(THRESHOLD_F1, color='green', ls='--', lw=1.5,
                label=f'Próg A={THRESHOLD_F1:.2f} (F1)')
axes[0].axvline(THRESHOLD_RECALL, color='red', ls='--', lw=1.5,
                label=f'Próg B={THRESHOLD_RECALL:.2f} (Recall)')
axes[0].set_xlabel('Risk score'); axes[0].set_ylabel('Density')
axes[0].set_title('Risk Score Distribution'); axes[0].legend(fontsize=8)
axes[0].grid(alpha=0.3)

# Cumulative: what % of leavers do we catch at a given TOP-N threshold?
n_pos = (df_risk['Attrition'] == 'Yes').sum()
sorted_labels = df_risk_sorted['Attrition'].values
cumulative_recall = np.cumsum(sorted_labels == 'Yes') / n_pos
axes[1].plot(range(1, len(cumulative_recall) + 1), cumulative_recall,
             color='darkorange', lw=2.5)
axes[1].axhline(0.70, color='red', ls='--', lw=1.5, label='Recall=0.70')
n70 = np.argmax(cumulative_recall >= 0.70) + 1
axes[1].axvline(n70, color='red', ls=':', lw=1.5, label=f'TOP {n70} catches 70% of leavers')
axes[1].axhline(0.80, color='purple', ls='--', lw=1.5, label='Recall=0.80')
n80 = np.argmax(cumulative_recall >= 0.80) + 1
axes[1].axvline(n80, color='purple', ls=':', lw=1.5, label=f'TOP {n80} catches 80% of leavers')
axes[1].set_xlabel('TOP-N employees by model'); axes[1].set_ylabel('Recall (% of leavers caught)')
axes[1].set_title('How many leavers do we catch in TOP-N interviews?')
axes[1].legend(fontsize=8); axes[1].grid(alpha=0.3)

print(f"To catch 70% of leavers → just talk to TOP {n70} ({n70/len(df_risk)*100:.1f}% of the company)")
print(f"To catch 80% of leavers → just talk to TOP {n80} ({n80/len(df_risk)*100:.1f}% of the company)")

plt.tight_layout()
plt.show()

---
## 10. HR Action Plan

### Threshold Selection – Decision for HR Manager

| Question | If YES → | If NO → |
|---|---|---|
| Do we have limited HR resources? | Threshold A (F1) – fewer interviews, higher precision | — |
| Is the cost of losing an employee high? | Threshold B (Recall) – do not miss anyone | — |
| Do we have a standing stay-interview program? | Threshold B – broad coverage | Threshold A |

### Step 1 – Verification with Managers (Priority: TOP 20)
Brief conversation with the manager: signs of dissatisfaction, overload, changes. Check if there was a recent promotion/raise – the model doesn't know this yet.

### Step 2 – "Stay Interview" 1:1 (without mentioning the model)
Narrative: "we want to better understand your needs". Questions:
- What do you like most / what bothers you at work?
- What could make you leave within 6–12 months?
- How can we support your development?

### Step 3 – Actions Based on Model Risk Factors
- **Compensation** → market review, raise plan, flexible bonuses
- **Overtime / WLB** → OT limits, flexible hours, remote work
- **Career** → promotion path, cross-team projects, 6–12 month development plan
- **Satisfaction** → team problem diagnosis, manager coaching

### Step 4 – Monitoring (after 3–6 months)
- How many of the TOP 20 are still with the company?
- Has the `risk_score` for these employees dropped on re-run?
- What conclusions → HR policies (overtime, raises, role rotation)

### Step 5 – Communication
- **To employees:** care for development, not a "risk list"
- **To management:** combining data analysis with retention actions – measurable ROI

####